# YOLO物体検出＋軌跡トラッキング

このノートブックでは、動画をアップロードするだけで物体検出と軌跡トラッキングを実行できます。

## 使い方
1. 上から順番にセルを実行してください（▶ボタンを押す）
2. 動画ファイルをアップロード
3. 処理完了後、ダウンロードボタンで結果を取得

## ステップ1: 環境セットアップ

In [ ]:
# 必要なライブラリをインストール
print("📦 ライブラリをインストール中...")
!pip install -q ultralytics opencv-python-headless
print("✅ インストール完了！")

# GPU確認
import torch
if torch.cuda.is_available():
    print(f"🚀 GPU利用可能: {torch.cuda.get_device_name(0)}")
else:
    print("⚠️ CPU モードで実行します（処理に時間がかかる場合があります）")

## ステップ2: リポジトリをクローン

スクリプトと重みファイルをGitHubから取得します。

In [ ]:
import os

# リポジトリをクローン（自分のリポジトリURLに変更してください）
REPO_URL = "https://github.com/koshien2015/ultralytics.git"  # ← ここを変更
BRANCH = "cap_detection"  # ブランチ名

print("📥 リポジトリをクローン中...")
if not os.path.exists("ultralytics"):
    !git clone -b {BRANCH} {REPO_URL}
    print("✅ クローン完了！")
else:
    print("✅ リポジトリは既に存在します")

# sharedディレクトリに移動
%cd ultralytics/shared
print(f"\n📂 現在のディレクトリ: {os.getcwd()}")
print(f"📄 ファイル一覧:")
!ls -lh *.py *.pt 2>/dev/null || echo "ファイルを確認中..."

## ステップ3: 動画ファイルをアップロード

検出対象の動画ファイル（.mp4など）をアップロードしてください。

In [ ]:
from google.colab import files

print("📹 動画ファイルをアップロードしてください")
video_uploaded = files.upload()
video_filename = list(video_uploaded.keys())[0]
print(f"✅ 動画アップロード完了: {video_filename}")

## ステップ4: 実行

このセルを実行すると、動画の処理が開始されます。

**機能:**
- 物体検出（ボール: クラス0）
- 軌跡トラッキング
- ピッチング解析（リリースポイント検出、ストライクゾーン推定、3D座標計算）
- カメラ角度推定

処理時間は動画の長さによって変わります（数分〜数十分）。

In [ ]:
print("🚀 処理を開始します...\n")

# track.pyを実行
import sys
import cv2
import os
import numpy as np
from ultralytics import YOLO
import tennis
from pitching_analysis import PitchingAnalyzer
import time

# 検出対象クラス（ボールのみ描画）
target_classes = [0]

# ファイル名とディレクトリを取得
base_name = os.path.splitext(os.path.basename(video_filename))[0]
video_dir = os.path.dirname(video_filename) if os.path.dirname(video_filename) else "."

# 強調動画のパスを設定
enhance_video = os.path.join(video_dir, f"{base_name}_enhance.mp4")

print(f"Original video: {video_filename}")
print(f"Generating enhanced video for detection...")

# tennisモジュールで強調動画を生成
tennis.run(video_filename, enhance_video_path=enhance_video)

print(f"\nLoading YOLO model...")
# YOLOモデルをロード（リポジトリに含まれる重みファイルを使用）
model = YOLO("yolo8m_20250510.pt")

# 軌跡描画設定
DRAW_TRAJECTORY = True
MAX_TRAJECTORY_LENGTH = 30
TRAJECTORY_FADE_FRAMES = 20
TRAJECTORY_COLOR = (0, 255, 0)
TRAJECTORY_THICKNESS = 4

# ピッチング解析設定
ENABLE_PITCHING_ANALYSIS = True
DRAW_STRIKE_ZONE = True
STRIKE_ZONE_WIDTH_PX = 50
STRIKE_ZONE_CENTER_X = None

trajectories = {}
next_object_id = 0

# ピッチング解析の初期化
analyzer = None
if ENABLE_PITCHING_ANALYSIS:
    print("Initializing pitching analyzer...")
    analyzer = PitchingAnalyzer(
        strike_zone_width_px=STRIKE_ZONE_WIDTH_PX,
        strike_zone_center_x=STRIKE_ZONE_CENTER_X
    )

# 強調動画と元動画を開く
cap_enhance = cv2.VideoCapture(enhance_video)
cap_original = cv2.VideoCapture(video_filename)

# 動画情報を取得
fps = cap_original.get(cv2.CAP_PROP_FPS)
width = int(cap_original.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap_original.get(cv2.CAP_PROP_FRAME_HEIGHT))

# 出力動画の設定
output_path = os.path.join(video_dir, f"{base_name}_detected.mp4")
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out = cv2.VideoWriter(output_path, fourcc, fps, (width, height))

# ピッチング解析にFPSを設定
if analyzer:
    analyzer.set_fps(fps)

print(f"\nProcessing frames and detecting objects...")

start_time = time.time()
frame_count = 0

while True:
    ret_enhance, frame_enhance = cap_enhance.read()
    ret_original, frame_original = cap_original.read()

    if not ret_enhance or not ret_original:
        break

    # 強調フレームで検出実行
    results = model(frame_enhance, verbose=False)

    # ピッチング解析
    if analyzer:
        analysis_result = analyzer.update(results, frame_count)

        # リリース検出時にログ出力
        if analysis_result['is_release']:
            print(f"🎯 Release detected at frame {frame_count}")

        # 描画
        frame_original = analyzer.draw(frame_original, frame_count,
                                      ball_3d=analysis_result['ball_3d'],
                                      draw_strike_zone=DRAW_STRIKE_ZONE,
                                      draw_info=True)

    # 現在フレームの検出物体の中心座標を取得
    current_centers = []

    # 検出結果を元フレームに描画
    for result in results:
        boxes = result.boxes
        for box in boxes:
            x1, y1, x2, y2 = map(int, box.xyxy[0])
            conf = float(box.conf[0])
            cls = int(box.cls[0])

            if target_classes and cls not in target_classes:
                continue

            center_x = int((x1 + x2) / 2)
            center_y = int((y1 + y2) / 2)
            current_centers.append((center_x, center_y, cls))

            cv2.rectangle(frame_original, (x1, y1), (x2, y2), (0, 255, 0), 2)
            label = f"{model.names[cls]} {conf:.2f}"
            cv2.putText(frame_original, label, (x1, y1-10),
                       cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2)

    # 軌跡を更新
    if DRAW_TRAJECTORY:
        used_centers = set()
        for obj_id in list(trajectories.keys()):
            if not current_centers:
                break

            points = trajectories[obj_id]['points']
            last_point = points[-1] if points else None
            if last_point is None:
                continue

            min_dist = float('inf')
            best_match = None
            for i, (cx, cy, cls) in enumerate(current_centers):
                if i in used_centers:
                    continue
                dist = np.sqrt((last_point[0] - cx)**2 + (last_point[1] - cy)**2)
                if dist < min_dist and dist < 100:
                    min_dist = dist
                    best_match = i

            if best_match is not None:
                cx, cy, cls = current_centers[best_match]
                trajectories[obj_id]['points'].append((cx, cy))
                trajectories[obj_id]['last_seen'] = frame_count
                used_centers.add(best_match)
                if len(trajectories[obj_id]['points']) > MAX_TRAJECTORY_LENGTH:
                    trajectories[obj_id]['points'].pop(0)

        for i, (cx, cy, cls) in enumerate(current_centers):
            if i not in used_centers:
                trajectories[next_object_id] = {
                    'points': [(cx, cy)],
                    'last_seen': frame_count
                }
                next_object_id += 1

        trajectories_to_remove = []
        for obj_id, traj_data in trajectories.items():
            if frame_count - traj_data['last_seen'] > TRAJECTORY_FADE_FRAMES:
                trajectories_to_remove.append(obj_id)
        for obj_id in trajectories_to_remove:
            del trajectories[obj_id]

        for obj_id, traj_data in trajectories.items():
            points = traj_data['points']
            frames_since_seen = frame_count - traj_data['last_seen']
            fade_alpha = max(0, 1 - (frames_since_seen / TRAJECTORY_FADE_FRAMES))

            if len(points) > 1 and fade_alpha > 0:
                for i in range(1, len(points)):
                    alpha = (i / len(points)) * fade_alpha
                    thickness = max(1, int(TRAJECTORY_THICKNESS * alpha))
                    color = tuple(int(c * fade_alpha) for c in TRAJECTORY_COLOR)
                    cv2.line(frame_original, points[i-1], points[i], color, thickness)

    out.write(frame_original)
    frame_count += 1
    if frame_count % 30 == 0:
        print(f"Processed {frame_count} frames")

cap_enhance.release()
cap_original.release()
out.release()

# 処理時間の計算
end_time = time.time()
processing_time = end_time - start_time
video_duration = frame_count / fps if fps > 0 else 0
processing_speed = video_duration / processing_time if processing_time > 0 else 0

print(f"\n{'='*60}")
print(f"Detection completed!")
print(f"{'='*60}")
print(f"Output saved to: {output_path}")
print(f"\n【Processing Statistics】")
print(f"  Total frames: {frame_count}")
print(f"  Video duration: {video_duration:.2f} seconds")
print(f"  Processing time: {processing_time:.2f} seconds")
print(f"  Processing speed: {processing_speed:.2f}x realtime")

# ピッチング解析のJSON出力
if analyzer:
    json_output_path = os.path.join(video_dir, f"{base_name}_trajectory.json")
    analyzer.export_to_json(json_output_path, video_file=video_filename)
    print(f"\nTrajectory data saved to: {json_output_path}")

print("\n🎉 処理完了！")

## ステップ5: 結果動画をダウンロード

処理が完了したら、このセルを実行して結果動画をダウンロードします。

In [ ]:
from google.colab import files

print("📥 結果動画をダウンロード中...")
files.download(output_path)
print("✅ ダウンロード完了！")

## ステップ6: 軌跡データ(JSON)をダウンロード

投球軌跡の3D座標データをJSONで取得します。
このファイルをThree.js + Next.jsで可視化することで、様々な視点から投球を分析できます。

In [ ]:
from google.colab import files
import os

# 軌跡データ(JSON)をダウンロード
if 'json_output_path' in globals() and os.path.exists(json_output_path):
    print(f"📥 軌跡データ(JSON)をダウンロード中: {json_output_path}")
    files.download(json_output_path)
    print("✅ ダウンロード完了！")
    print("\n💡 このJSONファイルはThree.js + Next.jsで3D可視化できます")
    print("   詳細: TRAJECTORY_VISUALIZATION.md を参照してください")
else:
    print("⚠️ 軌跡データが見つかりません")

## ステップ7 (オプション): 強調動画をダウンロード

動画前処理で生成された強調動画も確認したい場合は、このセルを実行してください。

In [ ]:
import os
from google.colab import files

if os.path.exists(enhance_video):
    print(f"📥 強調動画をダウンロード中: {enhance_video}")
    files.download(enhance_video)
    print("✅ ダウンロード完了！")
else:
    print("強調動画が見つかりません")